# 🧠 Notebook 03: Model Training (FAST Version)
## EfficientNet-B0 with Quadratic Weighted Kappa Optimization

**Optimizations Applied:**
- ✅ Cached preprocessing (Ben Graham + CLAHE) → 10-50x speedup
- ✅ Robust paths (auto-detect project root)
- ✅ Fixed DataLoader settings for MPS/CUDA
- ✅ Accurate timing instrumentation with device sync
- ✅ AMP only on CUDA (not MPS)

**Class Imbalance Fixes:**
- ✅ **WeightedRandomSampler** → Balances each batch (oversamples rare grades)
- ✅ **Focal Loss** → Focuses training on hard examples
- ✅ **Class Weights** → Penalizes misclassifying rare grades more


**Expected Results:**---

- Epoch time: ~2-5 minutes (vs. hours before)
- Val Kappa ≥ 0.82

## 1. Setup & Imports

In [ ]:
# Core imports
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Model architecture
import timm

# Image processing
import cv2
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Data
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix

# Visualization
import matplotlib.pyplot as plt

# Progress bar
from tqdm.notebook import tqdm

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"✅ Using device: {device}")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ timm version: {timm.__version__}")

## 2. Robust Path Configuration

Auto-detect project root by searching for marker directories.

In [ ]:
def find_project_root(start_path: Path = None) -> Path:
    """
    Find project root by searching upward for marker directories.
    Looks for a directory containing both 'src/' and 'notebooks/'.
    """
    if start_path is None:
        # Try to get notebook path, fallback to cwd
        try:
            start_path = Path(os.getcwd())
        except:
            start_path = Path.cwd()
    
    current = start_path.resolve()
    for _ in range(10):  # Max 10 levels up
        if (current / "src").exists() and (current / "notebooks").exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    
    raise RuntimeError(
        f"Could not find project root from {start_path}. "
        "Expected directory with 'src/' and 'notebooks/' subdirectories."
    )


def find_data_root(project_root: Path) -> Path:
    """
    Find the APTOS dataset directory by checking common locations.
    """
    candidates = [
        project_root / "aptos2019-blindness-detection",
        project_root.parent / "aptos2019-blindness-detection",
        project_root / "data" / "aptos2019-blindness-detection",
    ]
    
    for candidate in candidates:
        if candidate.exists() and (candidate / "train_images").exists():
            return candidate
    
    raise RuntimeError(
        f"Could not find APTOS dataset. Checked:\n" +
        "\n".join(f"  - {c}" for c in candidates)
    )


# Auto-detect paths
PROJECT_ROOT = find_project_root()
DATA_ROOT = find_data_root(PROJECT_ROOT)

# Directories
TRAIN_IMAGES_DIR = DATA_ROOT / "train_images"
SPLITS_DIR = PROJECT_ROOT / "splits"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
SRC_DIR = PROJECT_ROOT / "src"
CACHE_DIR = PROJECT_ROOT / "cache" / "preprocessed_224"

# Create directories if not exists
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Add src to path for imports (only once)
src_str = str(SRC_DIR)
if src_str not in sys.path:
    sys.path.insert(0, src_str)

# Load splits
df_train = pd.read_csv(SPLITS_DIR / "train.csv")
df_val = pd.read_csv(SPLITS_DIR / "val.csv")
df_test = pd.read_csv(SPLITS_DIR / "test.csv")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Data Root: {DATA_ROOT}")
print(f"📁 Cache Dir: {CACHE_DIR}")
print(f"📊 Training samples: {len(df_train):,}")
print(f"📊 Validation samples: {len(df_val):,}")
print(f"📊 Test samples: {len(df_test):,}")

## 3. Path Sanity Check

Verify all required paths exist before proceeding.

In [ ]:
print("="*70)
print("🔍 PATH SANITY CHECK")
print("="*70)

# Check critical paths
paths_to_check = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "DATA_ROOT": DATA_ROOT,
    "TRAIN_IMAGES_DIR": TRAIN_IMAGES_DIR,
    "SPLITS_DIR": SPLITS_DIR,
    "train.csv": SPLITS_DIR / "train.csv",
    "val.csv": SPLITS_DIR / "val.csv",
    "test.csv": SPLITS_DIR / "test.csv",
    "SRC_DIR": SRC_DIR,
}

all_ok = True
for name, path in paths_to_check.items():
    exists = path.exists()
    status = "✅" if exists else "❌"
    print(f"{status} {name}: {path}")
    if not exists:
        all_ok = False

# Check cache status
cached_files = list(CACHE_DIR.glob("*.npy")) if CACHE_DIR.exists() else []
cache_status = "✅" if len(cached_files) > 0 else "⚠️ (need to run caching)"
print(f"\n{cache_status} Cached files: {len(cached_files)}")

# Check sample images
sample_images = list(TRAIN_IMAGES_DIR.glob("*.png"))[:5]
print(f"\n📷 Sample images in train_images: {len(list(TRAIN_IMAGES_DIR.glob('*.png')))}")

assert all_ok, "❌ Some required paths are missing!"
print("\n" + "="*70)
print("✅ ALL PATH CHECKS PASSED!")
print("="*70)

## 4. Cache Preprocessing (Run Once)

This cell creates the cached preprocessed images. Run it once, then skip on subsequent runs.

In [ ]:
from preprocessing.preprocess import RetinaPreprocessor

def cache_images_if_needed(df, split_name, cache_dir, images_dir, img_size=224):
    """
    Cache preprocessed images if not already cached.
    """
    preprocessor = RetinaPreprocessor(
        img_size=img_size,
        ben_graham_sigma=10,
        clahe_clip=2.0,
        clahe_grid=(8, 8)
    )
    
    # Check how many need caching
    to_cache = []
    for idx, row in df.iterrows():
        img_id = row['id_code']
        cache_path = cache_dir / f"{img_id}.npy"
        if not cache_path.exists():
            to_cache.append((img_id, cache_path))
    
    if len(to_cache) == 0:
        print(f"✅ {split_name}: All {len(df)} images already cached")
        return
    
    print(f"⏳ {split_name}: Caching {len(to_cache)}/{len(df)} images...")
    
    for img_id, cache_path in tqdm(to_cache, desc=f"  {split_name}"):
        img_path = images_dir / f"{img_id}.png"
        
        # Preprocess: returns float32 [0,1]
        img_float = preprocessor.preprocess(
            img_path,
            apply_ben_graham=True,
            apply_clahe=True
        )
        
        # Convert to uint8 for storage efficiency
        img_u8 = (img_float * 255).clip(0, 255).astype(np.uint8)
        
        # Save
        np.save(cache_path, img_u8)
    
    print(f"✅ {split_name}: Caching complete!")


# Run caching for all splits
print("="*70)
print("🗄️  PREPROCESSING CACHE")
print("="*70)

cache_images_if_needed(df_train, "train", CACHE_DIR, TRAIN_IMAGES_DIR)
cache_images_if_needed(df_val, "val", CACHE_DIR, TRAIN_IMAGES_DIR)
cache_images_if_needed(df_test, "test", CACHE_DIR, TRAIN_IMAGES_DIR)

# Verify cache
cached_count = len(list(CACHE_DIR.glob("*.npy")))
expected_count = len(df_train) + len(df_val) + len(df_test)
print(f"\n📊 Cache status: {cached_count}/{expected_count} images cached")
print("="*70)

## 5. FastRetinaDataset (Cache-Based)

This dataset loads preprocessed images directly from cache, eliminating expensive preprocessing at training time.

In [ ]:
class FastRetinaDataset(Dataset):
    """
    Fast PyTorch Dataset that loads pre-cached preprocessed images.
    
    Features:
    - Loads cached uint8 .npy files directly (no preprocessing overhead)
    - Applies augmentations via Albumentations
    - ~10-50x faster than on-the-fly preprocessing
    """
    
    def __init__(
        self, 
        df: pd.DataFrame, 
        cache_dir: Path,
        transform=None,
    ):
        """
        Args:
            df: DataFrame with 'id_code' and 'diagnosis' columns
            cache_dir: Path to cached preprocessed images
            transform: Albumentations transform pipeline
        """
        self.df = df.reset_index(drop=True)
        self.cache_dir = Path(cache_dir)
        self.transform = transform
        
        # Verify cache exists
        if not self.cache_dir.exists():
            raise RuntimeError(f"Cache directory does not exist: {self.cache_dir}")
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['id_code']
        label = row['diagnosis']
        
        # Load cached image (uint8)
        cache_path = self.cache_dir / f"{img_id}.npy"
        
        if not cache_path.exists():
            raise FileNotFoundError(
                f"Cached image not found!\n"
                f"  ID: {img_id}\n"
                f"  Expected path: {cache_path}\n"
                f"  Run the caching cell first."
            )
        
        # Load uint8 image directly
        img = np.load(cache_path)  # Shape: (H, W, C), dtype: uint8
        
        # Apply augmentations if provided
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        else:
            # Default: convert to tensor and normalize
            img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        
        return img, torch.tensor(label, dtype=torch.long)


print("✅ FastRetinaDataset class created!")

## 6. Data Augmentations

In [ ]:
# ImageNet normalization values
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_train_transforms(img_size=224):
    """
    Training augmentations for fundus images.
    Expects uint8 input [0, 255].
    """
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=30, p=0.5, border_mode=cv2.BORDER_CONSTANT),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.1, 
            scale_limit=0.15, 
            rotate_limit=0, 
            p=0.5,
            border_mode=cv2.BORDER_CONSTANT
        ),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        ], p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=224):
    """
    Validation/Test transforms - only normalization.
    Expects uint8 input [0, 255].
    """
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

print("✅ Augmentation pipelines defined!")

## 7. Create DataLoaders (Fixed Settings)

In [ ]:
# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32

# ============================================================
# CLASS IMBALANCE ANALYSIS
# ============================================================
print("="*70)
print("⚖️  CLASS IMBALANCE ANALYSIS")
print("="*70)

class_counts = df_train['diagnosis'].value_counts().sort_index()
total_samples = len(df_train)

print("\n📊 Training Set Distribution:")
for grade, count in class_counts.items():
    pct = count / total_samples * 100
    bar = '█' * int(pct / 2)
    print(f"   Grade {grade}: {count:4d} ({pct:5.1f}%) {bar}")

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\n⚠️  Imbalance ratio: {imbalance_ratio:.1f}x (Grade 0 vs rarest class)")
print("   → Using WeightedRandomSampler to balance training!")
print("="*70)

# DataLoader settings based on device and environment
def is_notebook():
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            return True
    except:
        pass
    return False

IN_NOTEBOOK = is_notebook()

if IN_NOTEBOOK:
    NUM_WORKERS = 0
    PIN_MEMORY = False
elif device.type == 'mps':
    NUM_WORKERS = 2
    PIN_MEMORY = False
elif device.type == 'cuda':
    NUM_WORKERS = 4
    PIN_MEMORY = True
else:
    NUM_WORKERS = 2
    PIN_MEMORY = False

print(f"\n⚙️ DataLoader settings:")
print(f"   Device: {device}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Workers: {NUM_WORKERS} {'(notebook mode)' if IN_NOTEBOOK else ''}")
print(f"   Pin memory: {PIN_MEMORY}")

# Create datasets using cached images
train_dataset = FastRetinaDataset(
    df=df_train,
    cache_dir=CACHE_DIR,
    transform=get_train_transforms(IMG_SIZE),
)

val_dataset = FastRetinaDataset(
    df=df_val,
    cache_dir=CACHE_DIR,
    transform=get_val_transforms(IMG_SIZE),
)

# ============================================================
# WEIGHTED RANDOM SAMPLER (Key fix for class imbalance!)
# ============================================================
from torch.utils.data import WeightedRandomSampler

# Calculate sample weights (inverse of class frequency)
class_sample_counts = df_train['diagnosis'].value_counts().sort_index().values
class_weights_sampler = 1.0 / class_sample_counts
sample_weights = class_weights_sampler[df_train['diagnosis'].values]
sample_weights = torch.DoubleTensor(sample_weights)

# Create sampler
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True  # Required for oversampling minority classes
)

print(f"\n⚖️  WeightedRandomSampler created!")
print(f"   This oversamples minority classes (Grades 3,4) during training.")

# Build DataLoader kwargs
# NOTE: When using sampler, shuffle must be False
train_loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'sampler': train_sampler,  # Use sampler instead of shuffle!
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
}

val_loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'shuffle': False,
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
}

# Only add prefetch_factor and persistent_workers if using workers > 0
if NUM_WORKERS > 0:
    train_loader_kwargs['prefetch_factor'] = 2
    train_loader_kwargs['persistent_workers'] = True
    val_loader_kwargs['prefetch_factor'] = 2
    val_loader_kwargs['persistent_workers'] = True

# Create dataloaders
train_loader = DataLoader(train_dataset, **train_loader_kwargs)
val_loader = DataLoader(val_dataset, **val_loader_kwargs)

print(f"\n✅ DataLoaders created!")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

## 8. Timing Utilities

In [ ]:
def sync_device(device):
    """
    Synchronize device to get accurate timing measurements.
    Required for GPU/MPS where operations are asynchronous.
    """
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()
    # CPU is synchronous, no need to sync


class Timer:
    """Context manager for timing code blocks with device sync."""
    
    def __init__(self, name="", device=None):
        self.name = name
        self.device = device
        self.elapsed = 0
        
    def __enter__(self):
        if self.device:
            sync_device(self.device)
        self.start = time.perf_counter()
        return self
    
    def __exit__(self, *args):
        if self.device:
            sync_device(self.device)
        self.elapsed = time.perf_counter() - self.start


print("✅ Timing utilities defined!")

## 9. Sanity Check + Speed Test

Run 50 batches to verify everything works and measure actual timing.

In [ ]:
print("="*70)
print("🔍 SANITY CHECK + SPEED TEST")
print("="*70)

# ============================================================
# 1. Path validation
# ============================================================
print(f"\n📁 Resolved Paths:")
print(f"   PROJECT_ROOT: {PROJECT_ROOT}")
print(f"   DATA_ROOT: {DATA_ROOT}")
print(f"   TRAIN_IMAGES_DIR exists: {TRAIN_IMAGES_DIR.exists()}")

cached_files = list(CACHE_DIR.glob("*.npy"))
cached_count = len(cached_files)
print(f"   Cached files: {cached_count}")

# ============================================================
# 2. Cache validation
# ============================================================
print(f"\n🗄️  Cache Validation:")
if cached_count > 0:
    sample_cache_file = cached_files[0]
    sample_img = np.load(sample_cache_file)
    file_size_mb = sample_cache_file.stat().st_size / (1024 * 1024)
    
    print(f"   Sample file: {sample_cache_file.name}")
    print(f"   Shape: {sample_img.shape}")
    print(f"   Dtype: {sample_img.dtype}")
    print(f"   File size: {file_size_mb:.3f} MB")
    
    assert sample_img.shape == (224, 224, 3), f"❌ Wrong shape {sample_img.shape}"
    assert sample_img.dtype == np.uint8, f"❌ Wrong dtype {sample_img.dtype}"
    print(f"   ✅ Cache format validated!")
else:
    raise RuntimeError("❌ No cached files! Run caching cell first.")

# ============================================================
# 3. Create test model (EfficientNet-B0 for M1)
# ============================================================
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=False, num_classes=5)
    def forward(self, x):
        return self.backbone(x)

test_model = SimpleModel().to(device)
test_criterion = nn.CrossEntropyLoss()
test_optimizer = torch.optim.Adam(test_model.parameters(), lr=1e-4)

# ============================================================
# 4. Speed test - NO sync between ops (kills MPS performance)
# ============================================================
# IMPORTANT: On MPS, torch.mps.synchronize() is VERY expensive.
# We only sync once at the very end to measure total throughput.
# Component times are wall-clock approximations only.

NUM_TEST_BATCHES = 30  # Reduced for faster test
WARMUP_BATCHES = 3
test_model.train()

batch_times = []
print(f"\n⏱️ Speed test: {WARMUP_BATCHES} warmup + {NUM_TEST_BATCHES} measured batches...")

data_iter = iter(train_loader)

for batch_idx in tqdm(range(WARMUP_BATCHES + NUM_TEST_BATCHES), desc="Speed test"):
    batch_start = time.perf_counter()
    
    # Get batch
    try:
        images, labels = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        images, labels = next(data_iter)
    
    # Move to device
    images = images.to(device)
    labels = labels.to(device)
    
    # Forward
    outputs = test_model(images)
    loss = test_criterion(outputs, labels)
    
    # Backward
    test_optimizer.zero_grad()
    loss.backward()
    
    # Optimizer
    test_optimizer.step()
    
    batch_end = time.perf_counter()
    
    if batch_idx >= WARMUP_BATCHES:
        batch_times.append(batch_end - batch_start)

# Only sync ONCE at the end
if device.type == 'mps':
    torch.mps.synchronize()
elif device.type == 'cuda':
    torch.cuda.synchronize()

# ============================================================
# 5. Results
# ============================================================
avg_batch_time = np.mean(batch_times)
std_batch_time = np.std(batch_times)

print(f"\n📊 Timing Results (avg over {NUM_TEST_BATCHES} batches):")
print(f"-" * 50)
print(f"   Avg batch time: {avg_batch_time*1000:.1f} ms ± {std_batch_time*1000:.1f} ms")
print(f"   Throughput: {BATCH_SIZE / avg_batch_time:.1f} images/sec")

# Estimate epoch time
batches_per_epoch = len(train_loader)
estimated_epoch_minutes = (avg_batch_time * batches_per_epoch) / 60

print(f"\n⏱️ Estimated Training Performance:")
print(f"   Batches per epoch: {batches_per_epoch}")
print(f"   Estimated epoch time: {estimated_epoch_minutes:.1f} minutes")

# Cleanup
del test_model, test_optimizer, test_criterion

print(f"\n" + "="*70)
print("✅ SANITY CHECK + SPEED TEST PASSED!")
print("="*70)

## 10. Verify DataLoader Output

In [ ]:
# Verify dataloader output
print("🔍 Verifying dataloader output...")

batch_images, batch_labels = next(iter(train_loader))

print(f"\n✅ Batch verification:")
print(f"   Images shape: {batch_images.shape}")
print(f"   Labels shape: {batch_labels.shape}")
print(f"   Image dtype: {batch_images.dtype}")
print(f"   Value range: [{batch_images.min():.3f}, {batch_images.max():.3f}]")
print(f"   Labels: {batch_labels[:10].tolist()}")

## 11. Visualize Samples by Grade

Show samples from **each DR grade** to verify data diversity. Also verify that the WeightedRandomSampler produces more balanced batches.

In [ ]:
# Visualize samples - ONE FROM EACH GRADE
def denormalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Denormalize image tensor for visualization."""
    img = img_tensor.clone()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    return img.clamp(0, 1)

print("🔍 Collecting samples from each DR grade...")

# Create a temporary dataset without augmentation for cleaner visualization
vis_dataset = FastRetinaDataset(
    df=df_train,
    cache_dir=CACHE_DIR,
    transform=get_val_transforms(IMG_SIZE),  # No augmentation
)

# Find indices for each grade
grade_samples = {}
for grade in range(5):
    grade_indices = df_train[df_train['diagnosis'] == grade].index.tolist()
    if grade_indices:
        # Get 2 samples per grade
        for i, idx in enumerate(grade_indices[:2]):
            img, label = vis_dataset[idx]
            grade_samples[(grade, i)] = (img, label.item())

print(f"   Found samples for grades: {sorted(set(g for g, _ in grade_samples.keys()))}")

# Plot: 2 rows x 5 columns (one column per grade)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

grade_colors = ['green', 'yellowgreen', 'orange', 'orangered', 'red']
grade_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

for grade in range(5):
    for row in range(2):
        ax = axes[row, grade]
        key = (grade, row)
        if key in grade_samples:
            img_tensor, label = grade_samples[key]
            img = denormalize(img_tensor).permute(1, 2, 0).numpy()
            ax.imshow(img)
            if row == 0:
                ax.set_title(f"Grade {grade}\n({grade_names[grade]})", 
                           fontsize=12, fontweight='bold', color=grade_colors[grade])
        ax.axis('off')

plt.suptitle('Training Samples by DR Grade (Showing Data Diversity)', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'samples_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'samples_by_grade.png'}")

# Show batch distribution from balanced sampler
print("\n📊 Batch from WeightedRandomSampler (should be more balanced):")
batch_images, batch_labels = next(iter(train_loader))
label_counts = {i: (batch_labels == i).sum().item() for i in range(5)}
for grade, count in label_counts.items():
    bar = '█' * count
    print(f"   Grade {grade}: {count:2d} {bar}")

## 12. Build EfficientNet-B0 Model

In [ ]:
class RetinaModel(nn.Module):
    """
    EfficientNet-B0 based model for Diabetic Retinopathy grading.
    Optimized for Apple M1 - smaller and faster than B3.
    
    Features:
    - Pretrained EfficientNet-B0 backbone
    - Custom classification head with dropout
    - Supports feature extraction for GradCAM
    """
    
    def __init__(self, num_classes=5, pretrained=True, dropout_rate=0.3):
        super().__init__()
        
        # Load pretrained EfficientNet-B0
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=pretrained,
            num_classes=0,  # Remove original head
            global_pool=''  # We'll handle pooling ourselves
        )
        
        # Feature dimension: 1280 for B0
        self.num_features = self.backbone.num_features
        
        # Global average pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # Classification head
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(self.num_features, num_classes)
        )
        
    def forward_features(self, x):
        """Extract features before pooling (for GradCAM)."""
        return self.backbone(x)
    
    def forward(self, x):
        features = self.forward_features(x)
        pooled = self.global_pool(features)
        logits = self.head(pooled)
        return logits

# Create model
model = RetinaModel(num_classes=5, pretrained=True, dropout_rate=0.3)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ RetinaModel created!")
print(f"   Backbone: EfficientNet-B0")
print(f"   Feature dimension: {model.num_features}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

In [ ]:
# Test forward pass
print("🔍 Testing forward pass...")

with torch.no_grad():
    test_input = torch.randn(2, 3, 224, 224).to(device)
    test_output = model(test_input)

print(f"✅ Forward pass successful!")
print(f"   Input shape: {test_input.shape}")
print(f"   Output shape: {test_output.shape}")

## 13. Loss Function and Metrics

In [ ]:
def compute_class_weights(df, num_classes=5):
    """
    Compute class weights inversely proportional to class frequency.
    """
    class_counts = df['diagnosis'].value_counts().sort_index()
    total = len(df)
    
    weights = total / (num_classes * class_counts)
    weights = weights / weights.sum() * num_classes
    
    return torch.tensor(weights.values, dtype=torch.float32)

# Calculate class weights
class_weights = compute_class_weights(df_train)
print("📊 Class distribution in training set:")
print(df_train['diagnosis'].value_counts().sort_index())
print(f"\n⚖️ Computed class weights:")
for i, w in enumerate(class_weights):
    print(f"   Grade {i}: {w:.3f}")

In [ ]:
def quadratic_weighted_kappa(y_pred, y_true):
    """Calculate Quadratic Weighted Kappa (QWK)."""
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

def evaluate_model(model, dataloader, device):
    """
    Evaluate model on a dataloader.
    Returns dictionary with accuracy, kappa, and per-class metrics.
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    accuracy = accuracy_score(all_labels, all_preds)
    kappa = quadratic_weighted_kappa(all_preds, all_labels)
    conf_matrix = confusion_matrix(all_labels, all_preds)
    per_class_acc = conf_matrix.diagonal() / conf_matrix.sum(axis=1)
    
    return {
        'accuracy': accuracy,
        'kappa': kappa,
        'confusion_matrix': conf_matrix,
        'per_class_accuracy': per_class_acc,
        'predictions': all_preds,
        'labels': all_labels
    }

print("✅ Evaluation functions defined!")

## 14. Training Configuration

In [ ]:
# ============================================================
# FOCAL LOSS (Better for imbalanced data)
# ============================================================
class FocalLoss(nn.Module):
    """
    Focal Loss for handling class imbalance.
    Down-weights easy examples, focuses on hard ones.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: Class weights (tensor)
        gamma: Focusing parameter (default=2.0)
               gamma=0 → standard CE loss
               gamma>0 → reduces loss for well-classified examples
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)  # probability of correct class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

print("✅ FocalLoss class defined!")

# Training configuration
CONFIG = {
    'epochs': 15,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'patience': 5,
    'min_delta': 0.001,
    'focal_gamma': 2.0,      # Focal loss focusing parameter
}

# Loss function: Focal Loss with class weights
criterion = FocalLoss(
    alpha=class_weights.to(device),
    gamma=CONFIG['focal_gamma']
)

print(f"\n⚖️  Using Focal Loss with:")
print(f"   • Class weights: {[f'{w:.2f}' for w in class_weights.tolist()]}")
print(f"   • Gamma (focusing): {CONFIG['focal_gamma']}")

# Optimizer
optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])

# Learning rate scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=1e-6)

# AMP setup - ONLY for CUDA (not MPS)
use_amp = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler() if use_amp else None

print("\n✅ Training configuration:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")
print(f"\n   Optimizer: AdamW")
print(f"   Scheduler: CosineAnnealingLR")
print(f"   Loss: FocalLoss (weighted + gamma={CONFIG['focal_gamma']})")
print(f"   Sampler: WeightedRandomSampler (balanced batches)")
print(f"   AMP: {use_amp}")

## 15. Training Loop

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler=None):
    """
    Train model for one epoch with optional AMP.
    """
    model.train()
    running_loss = 0.0
    use_amp = scaler is not None
    
    pbar = tqdm(dataloader, desc="Training")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        if use_amp:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return running_loss / len(dataloader)

print("✅ Training function defined!")

In [ ]:
# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': [],
    'val_kappa': [],
    'lr': [],
    'epoch_time': [],
}

# Best model tracking
best_kappa = -1
best_epoch = 0
patience_counter = 0

print("="*70)
print("🚀 STARTING TRAINING (FAST VERSION)")
print("="*70)
print(f"Device: {device}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Learning Rate: {CONFIG['lr']}")
print(f"AMP Enabled: {use_amp}")
print("="*70 + "\n")

training_start_time = time.time()

for epoch in range(CONFIG['epochs']):
    sync_device(device)
    epoch_start = time.time()
    
    # Train
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    
    # Evaluate
    val_metrics = evaluate_model(model, val_loader, device)
    
    # Calculate validation loss
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
    val_loss /= len(val_loader)
    
    # Update scheduler
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    sync_device(device)
    epoch_time = time.time() - epoch_start
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_metrics['accuracy'])
    history['val_kappa'].append(val_metrics['kappa'])
    history['lr'].append(current_lr)
    history['epoch_time'].append(epoch_time)
    
    # Print progress
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']} ({epoch_time/60:.1f} min)")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Val Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"  Val Kappa: {val_metrics['kappa']:.4f}")
    print(f"  LR: {current_lr:.6f}")
    
    # Save best model
    if val_metrics['kappa'] > best_kappa + CONFIG['min_delta']:
        best_kappa = val_metrics['kappa']
        best_epoch = epoch + 1
        patience_counter = 0
        
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_kappa': best_kappa,
            'config': CONFIG
        }
        torch.save(checkpoint, MODELS_DIR / 'efficientnet_b0_best.pth')
        print(f"  💾 New best model saved! (Kappa: {best_kappa:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement for {patience_counter} epochs")
    
    # Early stopping
    if patience_counter >= CONFIG['patience']:
        print(f"\n⚠️ Early stopping triggered at epoch {epoch+1}")
        break

total_time = time.time() - training_start_time

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)
print(f"Total training time: {total_time/60:.1f} minutes")
print(f"Average epoch time: {np.mean(history['epoch_time'])/60:.1f} minutes")
print(f"Best Kappa: {best_kappa:.4f} (Epoch {best_epoch})")
print(f"Model saved: {MODELS_DIR / 'efficientnet_b0_best.pth'}")
print("="*70)

## 16. Plot Training Curves

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
ax1 = axes[0, 0]
ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Kappa
ax2 = axes[0, 1]
ax2.plot(history['val_kappa'], label='Val Kappa', marker='o', color='green')
ax2.axhline(y=0.82, color='red', linestyle='--', label='Target (0.82)')
ax2.axvline(x=best_epoch-1, color='purple', linestyle=':', label=f'Best (Epoch {best_epoch})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Quadratic Weighted Kappa')
ax2.set_title('Validation Kappa (Primary Metric)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Accuracy
ax3 = axes[1, 0]
ax3.plot(history['val_accuracy'], label='Val Accuracy', marker='o', color='orange')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Accuracy')
ax3.set_title('Validation Accuracy', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Epoch time
ax4 = axes[1, 1]
ax4.plot([t/60 for t in history['epoch_time']], label='Epoch Time', marker='o', color='purple')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Time (minutes)')
ax4.set_title('Epoch Time', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.suptitle(f'Training Curves - Best Kappa: {best_kappa:.4f} (Epoch {best_epoch})', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'training_curves.png'}")

## 17. Evaluate on Validation Set

In [ ]:
# Load best model
checkpoint = torch.load(MODELS_DIR / 'efficientnet_b0_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']}")

# Evaluate
val_results = evaluate_model(model, val_loader, device)

print("\n" + "="*70)
print("📊 VALIDATION SET EVALUATION")
print("="*70)
print(f"Accuracy: {val_results['accuracy']:.4f} ({val_results['accuracy']*100:.1f}%)")
print(f"Quadratic Weighted Kappa: {val_results['kappa']:.4f}")
print(f"\nPer-class Accuracy:")
for i, acc in enumerate(val_results['per_class_accuracy']):
    print(f"  Grade {i}: {acc:.4f} ({acc*100:.1f}%)")
print("="*70)

In [ ]:
import seaborn as sns

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))

cm = val_results['confusion_matrix']
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

sns.heatmap(
    cm_normalized, 
    annot=True, 
    fmt='.2%',
    cmap='Blues',
    xticklabels=['Grade 0', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4'],
    yticklabels=['Grade 0', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4'],
    ax=ax
)

ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (Normalized)\nAccuracy: {val_results["accuracy"]:.1%} | Kappa: {val_results["kappa"]:.4f}', 
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'confusion_matrix_val.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'confusion_matrix_val.png'}")

## 18. Save Training History and Model Info

In [ ]:
# Save training history
history_file = ARTIFACTS_DIR / 'training_history.json'
with open(history_file, 'w') as f:
    json.dump(history, f, indent=2)

# Save model info
model_info = {
    'model_name': 'EfficientNet-B0',
    'best_epoch': best_epoch,
    'best_kappa': float(best_kappa),
    'final_val_accuracy': float(val_results['accuracy']),
    'config': CONFIG,
    'training_samples': len(df_train),
    'validation_samples': len(df_val),
    'per_class_accuracy': val_results['per_class_accuracy'].tolist(),
    'total_training_time_minutes': total_time / 60,
    'avg_epoch_time_minutes': np.mean(history['epoch_time']) / 60,
    'timestamp': datetime.now().isoformat()
}

model_info_file = ARTIFACTS_DIR / 'model_info.json'
with open(model_info_file, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"✅ Saved training history: {history_file}")
print(f"✅ Saved model info: {model_info_file}")

## 19. Summary

In [ ]:
print("="*70)
print("📊 MODEL TRAINING COMPLETE - SUMMARY (FAST VERSION)")
print("="*70)
print(f"""
✅ Model Architecture: EfficientNet-B0 (optimized for M1)
   • Pretrained on ImageNet
   • Custom 5-class head with dropout (0.3)
   • Parameters: {total_params:,}

✅ Speed Optimizations Applied:
   • Cached preprocessing (Ben Graham + CLAHE)
   • Fixed DataLoader (workers={NUM_WORKERS}, pin_memory={PIN_MEMORY})
   • AMP: {use_amp}

✅ Training Results:
   • Best Epoch: {best_epoch}
   • Validation Accuracy: {val_results['accuracy']:.1%}
   • Validation Kappa: {val_results['kappa']:.4f}
   • Total Training Time: {total_time/60:.1f} minutes
   • Average Epoch Time: {np.mean(history['epoch_time'])/60:.1f} minutes

📁 Artifacts Generated:
   • {MODELS_DIR / 'efficientnet_b0_best.pth'}
   • {ARTIFACTS_DIR / 'training_curves.png'}
   • {ARTIFACTS_DIR / 'confusion_matrix_val.png'}
   • {ARTIFACTS_DIR / 'augmented_samples_fast.png'}
   • {ARTIFACTS_DIR / 'training_history.json'}
   • {ARTIFACTS_DIR / 'model_info.json'}
""")
print("="*70)
print("\n🚀 NEXT STEPS (Notebook 04):")
print("   1. Evaluate on sealed test set")
print("   2. Implement GradCAM explainability")
print("   3. Test-Time Augmentation (TTA)")
print("="*70)